<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/weeks/week-14/NB14_intentional_stance.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 14: Intentionality, the Intentional Stance and Can Machines Think?

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-14/lab.html>.

## Overview

Mental states are about things: beliefs are about the world and desires are for outcomes. This week asks whether machines can have states with this aboutness, examines Dennett's intentional stance and its formalisation as inverse planning, and closes the course by returning to the question of whether machines can think [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to explain intentionality and the distinction between original and derived intentionality, to describe the physical, design and intentional stances, to compute a goal posterior by inverse planning, and to defend a reasoned position on whether machines can think that draws on the whole course.

## The mark of the mental

Brentano proposed that what distinguishes mental phenomena is their intentionality, their being directed at an object [1]. A belief is about something, a desire is for something and a fear is of something, even when the object does not exist. Searle distinguished original from derived intentionality [4]. The words on a page and the states of a computer have derived intentionality: They are about things only because people interpret them. Minds have original intentionality, which does not depend on interpretation. On Searle's view, the symbols of a program have only derived intentionality, which connects this week with the Chinese Room of Week 4.

## The intentional stance

Dennett proposed a different approach [2]. There are three stances from which the behaviour of a system can be predicted. The physical stance uses physics and the system's physical constitution. The design stance assumes that the system will behave as it was designed to behave. The intentional stance treats the system as a rational agent with beliefs and desires and predicts that it will act to further its goals in the light of its beliefs. A system is an intentional system if its behaviour is reliably and voluminously predictable from the intentional stance. Dennett argued that beliefs are real in the way that patterns are real: They capture regularities in behaviour that cannot be captured more economically from lower-level stances [5]. The view sits between the realism of Searle and the view that talk of beliefs is merely useful fiction.

## Attributing minds

People attribute intentions readily. Heider and Simmel showed participants a short film of geometric shapes moving around a box, and almost all described the film in terms of intentions and emotions, such as a large triangle bullying a small one [6]. Baker, Saxe and Tenenbaum formalised such attributions as inverse planning [3]. An observer assumes that an agent acts approximately rationally towards a goal and uses Bayesian inference to work backwards from observed actions to the most probable goal. The model predicted human judgements of goals closely in their experiments. Figure 14.1 shows how the posterior over three goals sharpens as more of a path is observed. The lab uses this model and compares the predictions of the intentional stance with those of a simple physical stance.

![Figure 14.1](https://raw.githubusercontent.com/utkukose/philosophy-of-ai-NB-lecture/main/weeks/week-14/figures/w14_fig1.png)

*Figure 14.1. Posterior probabilities of three possible goals as successive steps of a path are observed, computed by Bayesian inverse planning [3].*

## Can machines think? The course in review

The intentional stance offers one answer to the question with which the course began. If adopting the stance towards a system yields reliable predictions, the system has beliefs and desires in the only sense that Dennett thinks matters, and many present AI systems qualify for a limited range of behaviour. Searle would reply that predictive usefulness does not create original intentionality, just as the Chinese Room produced appropriate behaviour without understanding. Shanahan's caution applies here too: The intentional stance towards a dialogue system may predict well while misleading about what the underlying model is [7].

The course has presented several ways to sharpen the question. Behavioural tests, such as Turing's, can be passed without settling it. Arguments about computation, grounding, systematicity, embodiment, relevance and limits bear on what thinking requires. Theories of consciousness address whether there is anything it is like to be the system. Studies of understanding, creativity and intentionality show where present systems succeed and where claims outrun evidence. The final essay asks each student to take a position and defend it with these tools.

> **Pause and reflect.** After fourteen weeks, has your answer to the question whether machines can think changed? Which week changed it most?

# Hands-on lab

The notebook implements Bayesian inverse planning in a grid, simulates goal-directed agents and compares how well the physical and intentional stances predict their next moves [2, 3].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Inverse planning

In [ ]:
GOALS = {"A": (6, 6), "B": (6, 0), "C": (0, 6)}
ACTS = {"N": (0, 1), "E": (1, 0), "S": (0, -1), "W": (-1, 0)}

def action_probs(pos, goal, beta):
    x, y = pos
    opts = {m: (x + dx, y + dy) for m, (dx, dy) in ACTS.items() if 0 <= x + dx <= 6 and 0 <= y + dy <= 6}
    util = np.array([-(abs(p[0] - goal[0]) + abs(p[1] - goal[1])) for p in opts.values()])
    w = np.exp(beta * (util - util.max()))
    return dict(zip(opts, w / w.sum()))

def goal_posterior(path, beta=1.5):
    logp = {g: 0.0 for g in GOALS}
    for a, b in zip(path[:-1], path[1:]):
        move = next(m for m, (dx, dy) in ACTS.items() if (a[0] + dx, a[1] + dy) == b)
        for g, pos in GOALS.items():
            logp[g] += np.log(action_probs(a, pos, beta)[move])
    v = np.array(list(logp.values()))
    p = np.exp(v - v.max())
    return dict(zip(GOALS, p / p.sum()))

print(goal_posterior([(0, 0), (1, 0), (2, 0), (3, 0)]))

### Your turn, exercise 1

Compute the posterior over goals after the path (0,0), (0,1), (1,1), (1,2), (2,2) with beta = 1.5, and store the probability of goal A in `p_goal_A`. Explain why A becomes the most probable goal even though the agent has moved towards C as much as towards B.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _posterior_reference():
    return goal_posterior([(0, 0), (0, 1), (1, 1), (1, 2), (2, 2)], 1.5)["A"]

In [ ]:
p_goal_A = None  # your computation
check("Exercise 1", p_goal_A, _posterior_reference())

## 2. Which stance predicts better?

Simulated agents choose a goal at random and move towards it with rationality 2.0. The physical stance predicts that the agent repeats its last move. The intentional stance infers the goal from the path so far and predicts the most probable next move.

In [ ]:
rng = np.random.default_rng(SEED)

def simulate(beta=2.0, steps=10):
    g = GOALS[rng.choice(list(GOALS))]
    path = [(0, 0)]
    for _ in range(steps):
        if path[-1] == g:
            break
        probs = action_probs(path[-1], g, beta)
        m = rng.choice(list(probs), p=list(probs.values()))
        path.append((path[-1][0] + ACTS[m][0], path[-1][1] + ACTS[m][1]))
    return path

def predict_intentional(path, beta=2.0):
    post = goal_posterior(path, beta)
    mix = {}
    for g, pg in post.items():
        for m, pm in action_probs(path[-1], GOALS[g], beta).items():
            mix[m] = mix.get(m, 0) + pg * pm
    return max(mix, key=mix.get)

def predict_physical(path):
    a, b = path[-2], path[-1]
    return next(m for m, (dx, dy) in ACTS.items() if (a[0] + dx, a[1] + dy) == b)

paths = [simulate() for _ in range(300)]

### Your turn, exercise 2

For every path and every step t from 2 to the end, predict the move from step t to t + 1 with both stances, using only the path up to step t. Return a tuple `(physical_accuracy, intentional_accuracy)`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _stances_reference():
    phys, intent = [], []
    for p in paths:
        for t in range(2, len(p)):
            actual = next(m for m, (dx, dy) in ACTS.items() if (p[t - 1][0] + dx, p[t - 1][1] + dy) == p[t])
            phys.append(predict_physical(p[:t]) == actual)
            intent.append(predict_intentional(p[:t]) == actual)
    return (float(np.mean(phys)), float(np.mean(intent)))

In [ ]:
stance_accuracy = None  # your computation
check("Exercise 2", stance_accuracy, _stances_reference())
print("Write three sentences: What does the comparison say about Dennett's criterion for being an intentional system?")

## Visual exploration: Goal inference step by step

An observer watches a path on a grid and updates its belief about which of three goals the agent pursues, assuming noisy rational movement. Edit the path to see how quickly the posterior commits.

In [ ]:
N = 7
goals = {"A": (6, 6), "B": (6, 0), "C": (0, 6)}

def probs(pos, g, beta=2.0):
    moves = [(0, 1), (1, 0), (0, -1), (-1, 0)]
    opts = [(m, (pos[0] + m[0], pos[1] + m[1])) for m in moves if 0 <= pos[0] + m[0] < N and 0 <= pos[1] + m[1] < N]
    u = np.array([-(abs(p[0] - g[0]) + abs(p[1] - g[1])) for _, p in opts]); w = np.exp(beta * (u - u.max()))
    return {m: wi for (m, _), wi in zip(opts, w / w.sum())}

path = [(3, 3), (4, 3), (4, 4), (5, 4), (5, 5), (6, 5), (6, 6)]
post, history = {g: 1 / 3 for g in goals}, []
history.append(dict(post))
for a, b in zip(path[:-1], path[1:]):
    m = (b[0] - a[0], b[1] - a[1])
    post = {g: post[g] * probs(a, goals[g])[m] for g in goals}
    z = sum(post.values()); post = {g: v / z for g, v in post.items()}
    history.append(dict(post))
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(*zip(*path), "o-")
for g, xy in goals.items():
    ax[0].annotate(g, xy, fontsize=13, color="red")
ax[0].set_xlim(-0.5, N - 0.5); ax[0].set_ylim(-0.5, N - 0.5); ax[0].set_aspect("equal"); ax[0].grid(True); ax[0].set_title("Observed path")
for g in goals:
    ax[1].plot([h[g] for h in history], "o-", label=f"goal {g}")
ax[1].set_xlabel("steps observed"); ax[1].set_ylabel("posterior probability"); ax[1].legend(); ax[1].set_title("Inverse planning")
plt.tight_layout(); plt.show()

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-14/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Describe what you saw in Part A before revealing the rules. What does your description reveal about the intentional stance?
2. Do present AI assistants satisfy Dennett's criterion for intentional systems? For which behaviours, and where does the stance fail?
3. State and defend your final answer to the question whether machines can think, citing at least three weeks of the course.

## Weekly task and submission

Write the final essay of the course, of about 1500 words: Can machines think? Take a clear position, defend it with arguments from at least five weeks, address the strongest objection to your position and use at least one lab or notebook result as evidence. Cite all sources in square brackets and attach the notebook of this week with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Do language model agents have beliefs?.** Evaluate whether it is correct, useful or misleading to attribute beliefs and desires to language model agents, drawing on Dennett, Searle and Shanahan [2, 4, 7].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## Final capstone

This week closes the course with the [Final Capstone: Minds, Morals and Machines](https://github.com/utkukose/philosophy-of-ai-NB-lecture/blob/main/exams/final/README.md).

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the final capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 14, with the report and all code files attached or linked.

## References

[1] Brentano, F. (1874). *Psychologie vom empirischen Standpunkte [Psychology from an Empirical Standpoint]*. Duncker & Humblot.

[2] Dennett, D. C. (1987). *The Intentional Stance*. MIT Press.

[3] Baker, C. L., Saxe, R., & Tenenbaum, J. B. (2009). Action understanding as inverse planning. *Cognition*, 113(3), 329-349. <https://doi.org/10.1016/j.cognition.2009.07.005>

[4] Searle, J. R. (1983). *Intentionality: An Essay in the Philosophy of Mind*. Cambridge University Press.

[5] Dennett, D. C. (1991). Real patterns. *The Journal of Philosophy*, 88(1), 27-51. <https://doi.org/10.2307/2027085>

[6] Heider, F., & Simmel, M. (1944). An experimental study of apparent behavior. *The American Journal of Psychology*, 57(2), 243-259. <https://doi.org/10.2307/1416950>

[7] Shanahan, M. (2024). Talking about large language models. *Communications of the ACM*, 67(2), 68-79. <https://doi.org/10.1145/3624724>